In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



In [2]:
df = pd.read_csv('Train.csv')
df

,ID,Target
0,uGI8F9Er0c5XwdnX,By this publique Act and Instrument of protest
1,Dqt2gc3V5RJqmLE5,By this publique Act and Instrument of protest
2,eqF93vAKYa4aOeHr,By this public act and Instrument of protest b...
3,SEYEZ7fwp3MbYkkB,By this public act and Instrument of protest b...
4,sQjpjiUTThaMf4j6,By this public act and Instrument of protest b...
...,...,...
4093,K0I5shzXqoC7CcOP,By this public act and Instrument of protest Be
4094,TBl3UE2gqnWxNmqh,To all Christian People to whome these present...
4095,XzaMRNbKDwFTHIZY,ffebruary which will be in the yeare of our Lo...
4096,NQAD74cvanuAX01k,Lawfull Attorney his heyres or Execut^rs Adm^r...


In [3]:
df.shape


(4098, 2)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4098 entries, 0 to 4097
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID      4098 non-null   object
 1   Target  4098 non-null   object
dtypes: object(2)
memory usage: 64.2+ KB


In [5]:
df.duplicated().sum()/len(df)*100

np.float64(0.0)

In [6]:
df.isnull().sum()/len(df)*100

,0
ID,0.0
Target,0.0


In [7]:

import re


def clean_text(text):
    # convert to lowercase
    text = text.lower()
    # remove urls
    text = re.sub(r"http\S+", "", text)

    # remove HTML
    text = re.sub(r"<.*?>", "", text)

    # remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # remove digits
    text = re.sub(r"\d+", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text

df['cleaned_text'] = df['Target'].apply(clean_text)


## MODELLING USING NATURAL PROCESSING LANGUAGE (NLP)

In [8]:
import nltk
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

In [9]:
nltk.download('punkt_tab')
df["tokens"] = df["cleaned_text"].apply(word_tokenize)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [10]:
df["tokens"]

,tokens
0,"[by, this, publique, act, and, instrument, of,..."
1,"[by, this, publique, act, and, instrument, of,..."
2,"[by, this, public, act, and, instrument, of, p..."
3,"[by, this, public, act, and, instrument, of, p..."
4,"[by, this, public, act, and, instrument, of, p..."
...,...
4093,"[by, this, public, act, and, instrument, of, p..."
4094,"[to, all, christian, people, to, whome, these,..."
4095,"[ffebruary, which, will, be, in, the, yeare, o..."
4096,"[lawfull, attorney, his, heyres, or, execut, r..."


In [11]:
import string
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = stopwords.words('english')
stop_words += list(string.punctuation)
# print(stop_words)

def remove_stopwords(token_list):
    filtered = [word.lower() for word in token_list if word.lower() not in stop_words]
    return filtered

df['filtered_tokens'] = df['tokens'].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
from nltk.stem import PorterStemmer

# Initialize the Porter Stemmer
porter_stemmer = PorterStemmer()

def stem_tokens(token_list):
    stemmed_words = [porter_stemmer.stem(word) for word in token_list]
    return stemmed_words

df['stemmed_tokens'] = df['filtered_tokens'].apply(stem_tokens)

In [13]:
df["length"] = df["stemmed_tokens"].apply(len)

df["length"].describe()

,length
count,4098.000000
mean,6.596877
std,1.907172
min,0.000000
25%,5.000000
50%,6.000000
75%,8.000000
max,13.000000


In [ ]:
# Run this cell without changes
import matplotlib.pyplot as plt
import seaborn as sns

# Set up figure and axes
num_genres = df['Target'].nunique()
fig, axes = plt.subplots(nrows=num_genres, figsize=(12, 2 * num_genres))

# Empty dict to hold words that have already been plotted and their colors
plotted_words_and_colors = {}
# Establish color palette to pull from
# (If you get an error message about popping from an empty list, increase this #)
color_palette = sns.color_palette('cividis', n_colors=80) # Increased n_colors to prevent IndexError

# Creating a plot for each unique genre
data_by_genre = [y for _, y in df.groupby('Target', as_index=False)]
for idx, genre_df in enumerate(data_by_genre):
    # Find top 10 words in this genre
    all_words_in_genre = genre_df.stemmed_tokens.explode()
    top_10 = all_words_in_genre.value_counts()[:10]

    # Select appropriate colors, reusing colors if words repeat
    colors = []
    for word in top_10.index:
        if word not in plotted_words_and_colors:
            new_color = color_palette.pop(0)
            plotted_words_and_colors[word] = new_color
        colors.append(plotted_words_and_colors[word])

    # Select axes, plot data, set title
    ax = axes[idx]
    ax.bar(top_10.index, top_10.values, color=colors)
    ax.set_title(genre_df.iloc[0].Target.title())

fig.tight_layout()

## spliting the data set

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['stemmed_tokens'], df['Target'], test_size=0.2, random_state=42)